# Grade do ICMC como rede — criação do dataset

**SCC0252 – Visualização Computacional (2026/2) · Projeto de Exploração e Análise
Visual de Dados**

Este notebook cobre **apenas a criação do conjunto de dados**: consulta às fontes,
coleta, limpeza e organização em tabelas. Ele é o "código utilizado pelo grupo para
consultar as fontes e criá-lo", exigido na 1ª entrega.

## Fonte dos dados

Todos os dados vêm do **JúpiterWeb** (`uspdigital.usp.br/jupiterweb`), o sistema
acadêmico público da USP. Para cada disciplina, o Júpiter tem três páginas:

| Página | URL | O que extraímos |
|---|---|---|
| Disciplina | `obterDisciplina?sgldis=SIGLA` | nome, departamento, créditos, ementa, docentes responsáveis, bibliografia |
| Requisitos | `listarCursosRequisitos?coddis=SIGLA` | pré-requisitos por curso e período ideal |
| Oferecimento | `obterTurma?sgldis=SIGLA` | turmas, horários, professores, vagas e inscritos |

Além disso, usamos a lista de disciplinas de cada unidade e as **grades
curriculares** dos cursos (`listarGradeCurricular`).

O parsing (a leitura do HTML) das três páginas de disciplina é feito pela biblioteca
open-source [`jupiterweb-scraper`](https://pypi.org/project/jupiterweb-scraper/)
(IME Jr, licença MIT). Nós adicionamos: cache em disco, coleta paralela com
pausas, parsing das grades curriculares, expansão para disciplinas de outras
unidades, normalização em tabelas e validação.

## O que o notebook produz

```
dados/
├── cache_html/        páginas HTML baixadas (permite re-executar sem internet)
├── bruto/             dados brutos de cada disciplina, em JSON
└── processado/        tabelas finais em CSV
    ├── disciplinas.csv           1 linha por disciplina (nós do grafo)
    ├── requisitos.csv            1 linha por aresta requisito → disciplina
    ├── periodo_ideal.csv         período ideal de cada disciplina em cada curso
    ├── cursos.csv                cursos de graduação das unidades
    ├── grade_curricular.csv      disciplina × curso (obrigatória/optativa, semestre)
    ├── docentes.csv              docentes únicos
    ├── disciplina_docente.csv    quem é responsável por / ministra cada disciplina
    ├── oferecimentos.csv         turmas do semestre corrente
    ├── horarios.csv              horários de cada turma
    ├── vagas.csv                 vagas, inscritos e matriculados por turma e curso
    └── bibliografia.csv          referências bibliográficas de cada disciplina
```

## Como executar

1. Rode as células em ordem (`Run All`). A primeira execução faz alguns milhares de
   requisições e leva de **10 a 30 minutos**; as seguintes usam o cache e levam
   segundos.
2. Se a execução for interrompida, basta rodar de novo: o que já foi baixado não é
   baixado outra vez.
3. Para atualizar os dados (por exemplo, no começo de um novo semestre), apague a
   pasta `dados/cache_html/`.

## 0. Dependências e configuração

In [1]:
# Instala as dependências (só é necessário na primeira vez).
%pip install -q jupiterweb-scraper==1.1.2 pandas networkx tqdm beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import hashlib
import importlib
import json
import re
import threading
import time
import unicodedata
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import networkx as nx
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm.auto import tqdm

from jupiterweb import Disciplina, Instituto

c:\Users\victo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Parâmetros da coleta

- **`UNIDADES`**: códigos das unidades cujas disciplinas formam o núcleo do dataset.
  O ICMC é a unidade `55`. A unidade `97` (EESC/ICMC) é a do curso interunidades de
  Engenharia de Computação, que usa muitas disciplinas do ICMC.
- **`PROFUNDIDADE_MAX_EXPANSAO`**: quantos "níveis" de disciplinas de **outras**
  unidades buscar (explicado na Etapa 4).
- **`N_WORKERS`** e **`PAUSA_S`**: controlam a "educação" do scraper. Com 4
  requisições simultâneas e uma pausa de 0,3 s depois de cada uma, o servidor da USP
  recebe no máximo ~10 requisições por segundo. Não aumentem muito esses valores: o
  Júpiter é usado por todos os alunos da USP.
- **`USER_AGENT`**: identifica a coleta para quem administra o servidor. É uma
  prática de boa conduta em scraping.

In [3]:
UNIDADES = {
    "55": "ICMC",
    "97": "EESC/ICMC (Engenharia de Computação)",
}
PROFUNDIDADE_MAX_EXPANSAO = 3
N_WORKERS = 4
PAUSA_S = 0.3
USER_AGENT = (
    "SCC0252-projeto-visualizacao/1.0 "
    "(coleta academica ICMC-USP; contato: victormoreli@usp.br)"
)

URL_BASE = "https://uspdigital.usp.br/jupiterweb/"

# Os notebooks ficam em notebooks/ e os dados em ../dados. Rodando da raiz
# do repositório, usa dados/.
DIR_DADOS = Path("../dados") if Path("../dados").exists() else Path("dados")
DIR_CACHE = DIR_DADOS / "cache_html"
DIR_BRUTO = DIR_DADOS / "bruto"
DIR_FINAL = DIR_DADOS / "processado"
for pasta in (DIR_CACHE, DIR_BRUTO, DIR_FINAL):
    pasta.mkdir(parents=True, exist_ok=True)

# A data da coleta é a do dia em que o cache foi criado, e não a da execução:
# re-executar o notebook sobre o cache não muda os dados. Ela fica gravada dentro
# da pasta do cache; apagar o cache também apaga a data.
ARQ_DATA_COLETA = DIR_CACHE / "_data_coleta.txt"
if ARQ_DATA_COLETA.exists():
    DATA_COLETA = ARQ_DATA_COLETA.read_text(encoding="utf-8").strip()
else:
    DATA_COLETA = date.today().isoformat()
    ARQ_DATA_COLETA.write_text(DATA_COLETA, encoding="utf-8")

print("Data da coleta:", DATA_COLETA)

Data da coleta: 2026-10-31


## 1. Camada de acesso: download com cache em disco

Toda página passa por uma única função, `obter_soup_cache(url)`, que faz três coisas:

1. **Cache.** Pensem no cache como uma "xerox" de cada página: a primeira vez que
   precisamos dela, vamos até a biblioteca (o servidor) e tiramos uma cópia; nas
   vezes seguintes, consultamos a cópia.
   *Formalmente:* o cache é uma memoização persistente da função
   `baixar: URL → bytes`. Ele mantém um mapa `C` (aqui, arquivos em disco nomeados
   pelo hash SHA-1 da URL). Ao pedir `u`: se `u ∈ C`, devolvemos `C[u]` sem acessar a
   rede; caso contrário, calculamos `b = baixar(u)`, armazenamos `C[u] = b` e
   devolvemos `b`.
   Isso torna o notebook **reprodutível** (re-execuções produzem exatamente o
   mesmo dataset) e **retomável** (se cair a internet no meio, nada se perde).
2. **Novas tentativas com espera exponencial** (`tenacity`). Se uma requisição falhar,
   esperamos 1 s, 2 s, 4 s… (até 30 s) e tentamos de novo, no máximo 5 vezes.
3. **Decodificação correta.** O Júpiter serve páginas em ISO-8859-1 (Latin-1), não em
   UTF-8. Sem isso, "Computação" viraria "ComputaÃ§Ã£o".

Por fim, **substituímos** a função de download interna da biblioteca
`jupiterweb-scraper` pela nossa (técnica conhecida como *monkeypatching*). Assim, todo
o parsing da biblioteca passa automaticamente pelo nosso cache.

In [4]:
_local = threading.local()


def _sessao():
    # Uma sessão HTTP por thread: reaproveita conexões e é seguro com paralelismo.
    if not hasattr(_local, "sessao"):
        _local.sessao = requests.Session()
        _local.sessao.headers["User-Agent"] = USER_AGENT
    return _local.sessao


def _arquivo_cache(url: str) -> Path:
    return DIR_CACHE / (hashlib.sha1(url.encode()).hexdigest() + ".html")


@retry(stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=30),
       reraise=True)
def _baixar(url: str) -> bytes:
    resposta = _sessao().get(url, timeout=20)
    resposta.raise_for_status()
    time.sleep(PAUSA_S)  # pausa de cortesia com o servidor
    return resposta.content


def obter_soup_cache(url: str) -> BeautifulSoup:
    arquivo = _arquivo_cache(url)
    if arquivo.exists():
        conteudo = arquivo.read_bytes()
    else:
        conteudo = _baixar(url)
        # Escreve num arquivo temporário e renomeia: se o processo for
        # interrompido no meio da escrita, o cache não fica com arquivo corrompido.
        temporario = arquivo.with_suffix(f".tmp{threading.get_ident()}")
        temporario.write_bytes(conteudo)
        temporario.replace(arquivo)
    html = conteudo.decode("iso-8859-1", errors="replace")
    return BeautifulSoup(html, "html.parser")


# Monkeypatching: a biblioteca passa a baixar páginas pela nossa função.
for modulo in ("jupiterweb.disciplina", "jupiterweb.instituto"):
    importlib.import_module(modulo).obter_soup = obter_soup_cache

print("Páginas já em cache:", len(list(DIR_CACHE.glob("*.html"))))

Páginas já em cache: 1903


## 2. Etapa 1 — Lista de disciplinas de cada unidade

A página `jupDisciplinaLista?codcg=55&letra=0-Z&tipo=D` lista todas as disciplinas
ativas do ICMC. Os prefixos das siglas indicam o departamento:

| Prefixo | Departamento |
|---|---|
| SCC | Ciências de Computação |
| SSC | Sistemas de Computação |
| SMA | Matemática |
| SME | Matemática Aplicada e Estatística |

Guardamos também **de qual listagem** cada sigla veio, porque isso vira um atributo
útil (`unidades_listagem`).

In [5]:
unidades_por_sigla = defaultdict(set)

for codigo, nome in UNIDADES.items():
    instituto = Instituto(codigo, nome)
    siglas = [d.sigla for d in instituto.obter_disciplinas()]
    for sigla in siglas:
        unidades_por_sigla[sigla].add(codigo)
    print(f"Unidade {codigo} ({nome}): {len(siglas)} disciplinas")

siglas_nucleo = sorted(unidades_por_sigla)
print("Total de disciplinas distintas no núcleo:", len(siglas_nucleo))

prefixos = pd.Series([s[:3] for s in siglas_nucleo]).value_counts()
prefixos.head(10)

Unidade 55 (ICMC): 450 disciplinas
Unidade 97 (EESC/ICMC (Engenharia de Computação)): 3 disciplinas
Total de disciplinas distintas no núcleo: 446


SSC    130
SME    108
SCC    102
SMA     99
550      4
970      3
Name: count, dtype: int64

## 3. Etapa 2 — Coleta detalhada de cada disciplina

Para cada sigla, `Disciplina(sigla).obter_dados()` acessa as três páginas (disciplina,
requisitos e oferecimento) e devolve um dicionário. Dois detalhes:

- A biblioteca devolve alguns campos como **objetos Python** (`Requisito`,
  `Oferecimento`, `HorarioAula`). A função `serializar_disciplina` os converte em
  dicionários e listas simples, para podermos salvar tudo em JSON.
- As **chaves** do dicionário são os títulos das seções da página, normalizados
  (minúsculas, sem acento): `"creditos aula"`, `"objetivos"`, `"bibliografia"` etc.
  Como cada disciplina mostra só as seções que se aplicam a ela, disciplinas
  diferentes podem ter chaves diferentes. A normalização em tabelas (Etapa 5) trata
  isso.

A coleta roda em paralelo (`ThreadPoolExecutor`). Disciplinas que falharem ficam
registradas em `falhas` em vez de interromper tudo. Os dados brutos são salvos em
`dados/bruto/disciplinas_brutas.json`, com a sigla como chave. Um dicionário vazio
`{}` indica que a disciplina não foi encontrada no Júpiter.

In [6]:
ARQ_BRUTO = DIR_BRUTO / "disciplinas_brutas.json"


def serializar_disciplina(sigla: str) -> dict:
    dados = Disciplina(sigla).obter_dados()
    if not dados:
        return {}  # sigla inexistente ou desativada
    saida = {}
    for chave, valor in dados.items():
        if chave == "requisitos":
            saida[chave] = {
                curso: [[{"sigla": r.sigla, "tipo": r.tipo} for r in grupo]
                        for grupo in grupos]
                for curso, grupos in valor.items()
            }
        elif chave == "oferecimento":
            saida[chave] = [{
                "turma": o.codigo,
                "data_inicio": o.data_inicio,
                "data_fim": o.data_fim,
                "tipo_turma": o.tipo_turma,
                "observacoes": o.observacoes,
                "horarios": [{
                    "dia_semana": h.dia_semana,
                    "hora_inicio": h.hora_inicio,
                    "hora_fim": h.hora_fim,
                    "professor": h.professor,
                } for h in o.horarios],
                "vagas": o.vagas,
            } for o in valor]
        else:
            saida[chave] = valor
    return saida


def coletar(siglas, bruto: dict, descricao: str) -> dict:
    # Coleta em paralelo as siglas que ainda não estão em `bruto`.
    pendentes = [s for s in siglas if s not in bruto]
    falhas = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futuros = {executor.submit(serializar_disciplina, s): s
                   for s in pendentes}
        for futuro in tqdm(as_completed(futuros), total=len(futuros),
                           desc=descricao):
            sigla = futuros[futuro]
            try:
                bruto[sigla] = futuro.result()
            except Exception as erro:  # noqa: BLE001
                falhas[sigla] = repr(erro)
    ARQ_BRUTO.write_text(json.dumps(bruto, ensure_ascii=False, indent=1),
                         encoding="utf-8")
    return falhas


bruto = (json.loads(ARQ_BRUTO.read_text(encoding="utf-8"))
         if ARQ_BRUTO.exists() else {})
falhas = coletar(siglas_nucleo, bruto, "Disciplinas do núcleo")

print(f"Coletadas: {len(bruto)} | não encontradas: "
      f"{sum(1 for v in bruto.values() if not v)} | falhas: {len(falhas)}")
if falhas:
    print("Exemplos de falhas (rodem a célula de novo para tentar outra vez):")
    print(list(falhas.items())[:5])

Disciplinas do núcleo: 100%|██████████| 446/446 [00:12<00:00, 34.69it/s]


Coletadas: 446 | não encontradas: 0 | falhas: 0


Vale inspecionar uma disciplina conhecida para ver o formato bruto, por exemplo a
própria SCC0252:

In [7]:
exemplo = bruto.get("SCC0252", {})
for chave, valor in exemplo.items():
    resumo = str(valor).replace("\n", " ")
    print(f"{chave:>28} │ {resumo[:90]}")

                   instituto │ Instituto de Ciências Matemáticas e de Computação
                departamento │ Ciências de Computação
                        nome │ Visualização Computacional
                 nome ingles │ Computational Visualization
               creditos aula │ 3
           creditos trabalho │ 1
         carga horaria total │ 75 h
                        tipo │ Semestral
                    ativacao │ 15/07/2023
                 desativacao │ 
                      ementa │ Introdução à Visualização e Analítica Visual de Dados e aplicações. Representações computa
                   objetivos │ O objetivo desta disciplina é fornecer uma introdução à área de Visualização de Dados e An
       conteudo programatico │ Introdução: Visualização, Mineração Visual e Analítica Visual de dados; aplicações em ciên
instrumentos e criterios de avaliacao │ {'metodo de avaliacao': 'Exposição de conceitos em aulas teóricas. Demonstrações de exempl
                bibliografia │ Liv

## 4. Etapa 3 — Cursos e grades curriculares

A página de requisitos informa o período ideal **somente** quando a disciplina tem
requisitos naquele curso. Disciplinas do 1º semestre, por exemplo, não aparecem lá.
Para saber, para cada curso, **quais disciplinas são obrigatórias ou optativas e em
que semestre ideal ficam**, precisamos da grade curricular.

1. `jupCursoLista?codcg=55&tipo=N` lista os cursos da unidade. Cada curso tem um link
   para `listarGradeCurricular?codcg=..&codcur=..&codhab=..`. `codcur` é o código do
   curso, e `codhab` é a habilitação ou ênfase (por exemplo, período integral ou
   noturno).
2. Na grade, percorremos as linhas da tabela em ordem, como quem lê de cima para
   baixo, e mantemos um **estado**:
   - linhas de cabeçalho com "Obrigatórias", "Optativas Eletivas" ou "Optativas
     Livres" mudam a **seção** atual (e zeram o semestre, para que o semestre de
     uma seção não "vaze" para a seguinte);
   - linhas como "3º Período Ideal" mudam o **semestre** atual. Esse cabeçalho fica
     na mesma linha que os títulos das colunas ("Créd. Aula", "Créd. Trab.", "CH"
     etc.), então ele é procurado em qualquer linha, independentemente do número de
     células;
   - linhas que começam com uma sigla e têm pelo menos duas colunas numéricas
     (créditos aula e trabalho) são disciplinas da grade. As linhas de requisitos que
     aparecem embaixo de cada disciplina não têm colunas de créditos e, por isso, são
     ignoradas aqui (os requisitos já vêm da Etapa 2).

> **Atenção:** esta é a única parte do parsing escrita por nós, e não pela
> biblioteca. A célula de verificação logo abaixo mostra quantas disciplinas foram
> encontradas em cada curso. Se algum curso aparecer com 0, o layout da página mudou:
> abram a URL do curso no navegador e ajustem as regras da função `parsear_grade`.

In [8]:
RE_SIGLA = re.compile(r"^[A-Z0-9]{3}\d{4}$")
# O cabeçalho do semestre ("1º Período Ideal") fica na mesma linha que os
# títulos das colunas ("Créd. Aula", "Créd. Trab.", "CH"...), com 8 células.
# Por isso ele é procurado em qualquer linha, e não só nas linhas curtas.
# Exigir "Período Ideal" evita casar com "8 semestres" (duração do curso).
RE_PERIODO = re.compile(r"(\d+)\s*[ºo°ª]?\s*Per[ií]odo\s+Ideal", re.I)


def listar_cursos(codcg: str) -> list[dict]:
    soup = obter_soup_cache(f"{URL_BASE}jupCursoLista?codcg={codcg}&tipo=N")
    cursos = {}
    for link in soup.select("a[href*='listarGradeCurricular']"):
        params = parse_qs(urlparse(link["href"]).query)
        codcur = params.get("codcur", [""])[0]
        codhab = params.get("codhab", ["0"])[0]
        if not codcur:
            continue
        cursos[(codcur, codhab)] = {
            "codcg": codcg,
            "codcur": codcur,
            "codhab": codhab,
            "nome_curso": " ".join(link.get_text(" ", strip=True).split()),
            "url_grade": urljoin(URL_BASE, link["href"]),
        }
    return list(cursos.values())


def _texto_celulas(linha) -> list[str]:
    celulas = [" ".join(td.get_text(" ", strip=True).split())
               for td in linha.find_all("td", recursive=False)]
    return [c for c in celulas if c]


def parsear_grade(soup: BeautifulSoup, curso: dict) -> list[dict]:
    linhas, secao, periodo = [], None, None
    for tr in soup.find_all("tr"):
        if tr.find("table"):
            continue  # linha "container" de tabelas aninhadas
        celulas = _texto_celulas(tr)
        if not celulas:
            continue
        texto = " ".join(celulas).lower()

        achado = RE_PERIODO.search(texto)
        if achado:
            periodo = int(achado.group(1))
            continue

        primeira = celulas[0].split()[0]
        numeros = [c for c in celulas[1:] if re.fullmatch(r"\d+", c)]
        if RE_SIGLA.match(primeira) and len(numeros) >= 2:
            nome = next((c for c in celulas[1:] if not c.isdigit()), "")
            linhas.append({
                "codcur": curso["codcur"],
                "codhab": curso["codhab"],
                "sigla": primeira,
                "nome_na_grade": nome,
                "tipo_vinculo": secao,
                "semestre_ideal": periodo,
                "creditos_aula": int(numeros[0]),
                "creditos_trabalho": int(numeros[1]),
            })
            continue

        # Cabeçalhos de seção: atualizam o tipo de vínculo.
        if len(celulas) <= 2:
            nova_secao = ("obrigatoria" if "obrigat" in texto else
                          "optativa_eletiva" if "eletiva" in texto else
                          "optativa_livre" if "livre" in texto else None)
            if nova_secao:
                # Nova seção: o semestre da seção anterior não vale mais.
                secao, periodo = nova_secao, None
    return linhas


cursos = [c for codcg in UNIDADES for c in listar_cursos(codcg)]
grade = []
for curso in tqdm(cursos, desc="Grades curriculares"):
    grade.extend(parsear_grade(obter_soup_cache(curso["url_grade"]), curso))

df_cursos = pd.DataFrame(cursos)
df_grade = (pd.DataFrame(grade)
            .drop_duplicates(["codcur", "codhab", "sigla"]))
print(f"{len(df_cursos)} cursos/habilitações | {len(df_grade)} linhas de grade")

Grades curriculares: 100%|██████████| 9/9 [00:00<00:00,  9.46it/s]

9 cursos/habilitações | 877 linhas de grade


In [9]:
# Verificação: quantas disciplinas foram lidas em cada curso, por tipo de vínculo.
if df_grade.empty:
    print("⚠ Nenhuma disciplina lida nas grades: revisem `parsear_grade`.")
else:
    resumo_grade = (df_grade
                    .groupby(["codcur", "codhab", "tipo_vinculo"], dropna=False)
                    .size().unstack(fill_value=0))
    resumo_grade = resumo_grade.join(
        df_cursos.set_index(["codcur", "codhab"])["nome_curso"])
    display(resumo_grade)
    sem_semestre = df_grade["semestre_ideal"].isna().sum()
    print(f"Linhas da grade sem semestre ideal: {sem_semestre} de {len(df_grade)}")

obrigatoria  optativa_eletiva  optativa_livre  \
codcur codhab                                                  
55030  100               5                 0               0   
       200              26                57               4   
       300              32                46               5   
55041  0                45                88               7   
55051  4                43                39               1   
55060  0                30                52               4   
55071  4                32                34              11   
55090  0                30                74               4   
97001  0                63               123              22   

                                                      nome_curso  
codcur codhab                                                     
55030  100                             Matemática - Núcleo Geral  
       200                             Bacharelado em Matemática  
       300                            Licenciatura em Matemática  
55041  0                   Bacharelado em Ciências de Computação  
55051  4                   Bacharelado em Sistemas de Informação  
55060  0       Bacharelado em Matemática Aplicada e Computaçã...  
55071  4           Bacharelado em Estatística e Ciência de Dados  
55090  0                         Bacharelado em Ciência de Dados  
97001  0                                Engenharia de Computação

Linhas da grade sem semestre ideal: 0 de 877


## 5. Etapa 4 — Expansão para disciplinas de outras unidades

Os cursos do ICMC dependem de disciplinas de outras unidades: física no IFSC,
eletrônica na EESC etc. Elas aparecem como **requisitos** ou na **grade**, mas não na
listagem do ICMC. Sem elas, o grafo teria arestas apontando para nós sem nome,
créditos ou departamento.

Imaginem puxar um fio de um novelo: pegamos as disciplinas que já temos, vemos quais
outras elas citam, buscamos essas, vemos quais *essas* citam, e assim por diante.

*Formalmente:* é uma **busca em largura (BFS)** sobre o grafo de referências. Seja
`V₀` o conjunto do núcleo (Etapas 1–2) e `ref(v)` o conjunto de siglas citadas por
`v` (seus requisitos, mais as siglas das grades). A cada nível `k`, calculamos a
fronteira `F_k = (⋃_{v ∈ V_k} ref(v)) ∖ V_k`, coletamos `F_k` e definimos
`V_{k+1} = V_k ∪ F_k`. O processo para quando `F_k = ∅` (fechamento completo) ou
quando `k` atinge `PROFUNDIDADE_MAX_EXPANSAO`. A coluna `profundidade` registra em
que nível cada disciplina entrou (0 = núcleo).

In [10]:
def siglas_citadas(dados: dict) -> set[str]:
    citadas = set()
    for grupos in dados.get("requisitos", {}).values():
        for grupo in grupos:
            citadas.update(r["sigla"] for r in grupo)
    return citadas


profundidade = {s: 0 for s in siglas_nucleo}
fronteira = set(df_grade["sigla"]) if not df_grade.empty else set()

for nivel in range(1, PROFUNDIDADE_MAX_EXPANSAO + 1):
    for sigla in list(profundidade):
        fronteira |= siglas_citadas(bruto.get(sigla, {}))
    fronteira = {s for s in fronteira if s not in profundidade}
    if not fronteira:
        print(f"Fechamento completo no nível {nivel - 1}.")
        break
    for sigla in fronteira:
        profundidade[sigla] = nivel
    falhas_exp = coletar(sorted(fronteira), bruto, f"Expansão nível {nivel}")
    print(f"Nível {nivel}: +{len(fronteira)} disciplinas "
          f"({len(falhas_exp)} falhas)")
    fronteira = set()

print("Total de disciplinas no dataset:", len(profundidade))

Expansão nível 1: 100%|██████████| 146/146 [00:04<00:00, 35.79it/s]


Nível 1: +146 disciplinas (0 falhas)


Expansão nível 2: 100%|██████████| 26/26 [00:00<00:00, 32.56it/s]


Nível 2: +26 disciplinas (0 falhas)


Expansão nível 3: 100%|██████████| 14/14 [00:00<00:00, 31.58it/s]


Nível 3: +14 disciplinas (0 falhas)
Total de disciplinas no dataset: 632


## 6. Etapa 5 — Normalização em tabelas

Agora transformamos o JSON bruto, aninhado e com chaves variáveis, em **tabelas
"tidy"**: cada linha é uma observação e cada coluna é uma variável. É o formato que
pandas, Altair, Plotly e D3 esperam.

Pensem num modelo relacional, como num banco de dados: em vez de uma tabela gigante
com colunas repetidas (`requisito_1`, `requisito_2`…), temos uma tabela de
**entidades** (disciplinas, docentes, cursos) e tabelas de **relações** (requisitos,
grade, disciplina–docente), ligadas por chaves (`sigla`, `codcur`).

*Formalmente:* as tabelas estão na **1ª forma normal (1FN)**, em que todo atributo
é atômico (sem listas dentro de células) e cada linha é identificada por uma chave.
As tabelas de relação são **listas de arestas**: `requisitos.csv` define o grafo
direcionado `G = (V, E)`, com `V` = disciplinas e `(u, v) ∈ E` se `u` é requisito
de `v` em algum curso.

### Funções auxiliares

- `achatar`: transforma dicionários aninhados (como `avaliacao → criterio`) em
  chaves planas (`"avaliacao > criterio"`).
- `campo`: busca um campo pelas palavras contidas na chave. Isso torna o código
  tolerante a pequenas variações nos títulos das seções.
- `inteiro`: extrai o primeiro número de um texto (`"60 h"` → `60`).
- `normalizar_nome`: remove acentos e padroniza maiúsculas, para casar nomes de
  docentes que aparecem escritos de formas ligeiramente diferentes.

In [11]:
def achatar(d: dict, prefixo: str = "") -> dict:
    plano = {}
    for chave, valor in d.items():
        nome = f"{prefixo} > {chave}" if prefixo else chave
        if isinstance(valor, dict):
            plano.update(achatar(valor, nome))
        elif isinstance(valor, str):
            plano[nome] = valor
    return plano


def campo(plano: dict, *palavras, exato: str | None = None) -> str | None:
    # `exato`: nome completo da chave achatada (ex.: "ementa" ou
    # "instrumentos e criterios de avaliacao > metodo de avaliacao").
    if exato is not None:
        return plano.get(exato) or None
    # `palavras`: todas devem aparecer no último nível da chave.
    for chave, valor in plano.items():
        ultimo = chave.split(" > ")[-1]
        if all(p in ultimo for p in palavras):
            return valor or None
    return None


def inteiro(texto):
    achado = re.search(r"\d+", str(texto or ""))
    return int(achado.group()) if achado else None


# O Júpiter marca com "(R)" o professor responsável pela turma, colado ao nome
# ("(R) Fulano" ou "fulano (r)"). A marca é removida para não duplicar o docente.
RE_MARCA_R = re.compile(r"^\s*\(R\)\s*|\s*\(R\)\s*$", re.I)


def limpar_nome(nome: str) -> str:
    return " ".join(RE_MARCA_R.sub("", nome or "").split())


def normalizar_nome(nome: str) -> str:
    sem_acento = unicodedata.normalize("NFKD", limpar_nome(nome))
    sem_acento = sem_acento.encode("ascii", "ignore").decode()
    return " ".join(sem_acento.upper().split())


# Na página de requisitos o curso aparece como "55041 Bacharelado em ..." (sem
# hífen). O hífen fica opcional para aceitar também "55041 - Bacharelado ...".
RE_CURSO = re.compile(r"^\s*([\d/]+)\s*-?\s*(.+?)\s*$")


def separar_curso(texto: str) -> tuple[str, str]:
    # "55041 Bacharelado em Ciências de Computação" -> ("55041", "Bacharel...")
    achado = RE_CURSO.match(texto or "")
    if not achado:
        return "", (texto or "").strip()
    return achado.group(1).split("/")[0], achado.group(2)


def codhab_do_curso(codcur: str, nome_curso: str) -> str | None:
    # Um mesmo codcur pode ter várias habilitações (ex.: 55030 = Matemática,
    # com Bacharelado e Licenciatura). Nos requisitos, a habilitação só aparece no
    # nome ("Matemática - Habilitação: Licenciatura em Matemática"). Procuramos a
    # habilitação de df_cursos cujo nome está contido nesse texto.
    opcoes = df_cursos[df_cursos["codcur"] == codcur]
    if len(opcoes) == 1:
        return opcoes["codhab"].iloc[0]
    nome = normalizar_nome(nome_curso)
    for _, curso in opcoes.iterrows():
        if normalizar_nome(curso["nome_curso"]) in nome:
            return curso["codhab"]
    return None  # curso de outra unidade ou habilitação não identificada

### 6.1 `disciplinas` — os nós do grafo

Uma linha por disciplina, com atributos numéricos (créditos, carga horária,
contagens) e textuais (ementa, bibliografia). O texto fica guardado agora para a 2ª
entrega, em que planejamos calcular a similaridade das ementas.

In [12]:
registros = []
for sigla, dados in bruto.items():
    if not dados:
        continue
    plano = achatar({k: v for k, v in dados.items()
                     if k not in ("requisitos", "oferecimento", "periodo ideal")})
    docentes_txt = campo(plano, exato="docente(s) responsavel(eis)") or ""
    # Algumas disciplinas dividem a bibliografia em básica e complementar.
    partes_biblio = [campo(plano, exato=k) for k in (
        "bibliografia", "bibliografia basica", "bibliografia complementar")]
    biblio = "\n".join(p for p in partes_biblio if p) or None
    registros.append({
        "sigla": sigla,
        "nome": dados.get("nome"),
        "nome_ingles": dados.get("nome ingles"),
        "unidade": dados.get("instituto"),
        "departamento": dados.get("departamento"),
        "prefixo": sigla[:3] if sigla[:3].isalpha() else None,
        "creditos_aula": inteiro(campo(plano, exato="creditos aula")),
        "creditos_trabalho": inteiro(campo(plano, exato="creditos trabalho")),
        "carga_horaria_total": inteiro(campo(plano, exato="carga horaria total")),
        "tipo": campo(plano, exato="tipo"),
        "ativacao": campo(plano, exato="ativacao"),
        "desativacao": campo(plano, exato="desativacao"),
        "ementa": campo(plano, exato="ementa"),
        "objetivos": campo(plano, exato="objetivos"),
        "programa": campo(plano, exato="conteudo programatico"),
        "avaliacao_metodo": campo(plano, exato="instrumentos e criterios de avaliacao > metodo de avaliacao"),
        "avaliacao_criterio": campo(plano, exato="instrumentos e criterios de avaliacao > criterio de avaliacao"),
        "avaliacao_recuperacao": campo(plano, exato="instrumentos e criterios de avaliacao > norma de recuperacao"),
        "bibliografia_bruta": biblio,
        "docentes_responsaveis_bruto": docentes_txt or None,
        "n_turmas_oferecidas": len(dados.get("oferecimento", [])),
        "n_cursos_com_requisitos": len(dados.get("requisitos", {})),
        "unidades_listagem": ";".join(sorted(unidades_por_sigla.get(sigla, []))),
        "profundidade": profundidade.get(sigla),
        "no_nucleo": sigla in unidades_por_sigla,
        "url": f"{URL_BASE}obterDisciplina?sgldis={sigla}",
        "data_coleta": DATA_COLETA,
    })

df_disciplinas = (pd.DataFrame(registros)
                  .sort_values("sigla").reset_index(drop=True))
df_disciplinas["ano_ativacao"] = (df_disciplinas["ativacao"]
                                  .str.extract(r"(\d{4})")[0].astype("Int64"))
df_disciplinas["creditos_total"] = (df_disciplinas["creditos_aula"].fillna(0)
                                    + df_disciplinas["creditos_trabalho"].fillna(0))
print(df_disciplinas.shape)
df_disciplinas[["sigla", "nome", "departamento", "creditos_aula",
                "creditos_trabalho", "profundidade"]].head()

(629, 29)


,sigla,nome,departamento,creditos_aula,creditos_trabalho,profundidade
0,1800040,A Ética e a Responsabilidade Social em Engenharia,Disciplinas Interdepartamentais da EESC,4,0,1
1,1800043,O Engenheiro Como Agente Ético,Disciplinas Interdepartamentais da EESC,4,0,1
2,1800118,Introdução à Engenharia Biomédica,Disciplinas Interdepartamentais da EESC,2,0,1
3,1800119,Anatomia e Fisiologia para o Engenheiro Biomédico,Disciplinas Interdepartamentais da EESC,2,0,1
4,1800120,Tutoria Acadêmica I,Disciplinas Interdepartamentais da EESC,1,1,1


### 6.2 `requisitos` e `periodo_ideal` — as arestas do grafo

No Júpiter, os requisitos são definidos **por curso** e em **grupos alternativos**:
"para cursar X, alunos do curso C precisam de (A **e** B) **ou** (D)". Em lógica,
isso é uma expressão na **forma normal disjuntiva**: um "ou" de grupos, cada grupo
sendo um "e" de requisitos. Representamos cada requisito de cada grupo como uma
aresta e guardamos:

- `grupo_alternativa`: índice do grupo (0, 1, …);
- `n_grupos`: quantos grupos alternativos existem para aquele curso. Se for 1, a
  aresta é **obrigatória**; se for maior, ela é **um dos caminhos possíveis**;
- `tipo_requisito`: `requisito` (precisa ter sido aprovado), `requisito fraco`
  (basta ter cursado, mesmo com reprovação) ou `indicação de conjunto` (as
  disciplinas devem ser cursadas juntas).

A direção da aresta é `requisito → disciplina`, ou seja, "vem antes de".

In [13]:
arestas, periodos = [], []
for sigla, dados in bruto.items():
    for curso_txt, grupos in dados.get("requisitos", {}).items():
        codcur, nome_curso = separar_curso(curso_txt)
        for indice, grupo in enumerate(grupos):
            for req in grupo:
                arestas.append({
                    "requisito": req["sigla"],
                    "disciplina": sigla,
                    "tipo_requisito": req["tipo"],
                    "codcur": codcur,
                    "codhab": codhab_do_curso(codcur, nome_curso),
                    "nome_curso": nome_curso,
                    "grupo_alternativa": indice,
                    "n_grupos": len(grupos),
                })
    for curso_txt, periodo in dados.get("periodo ideal", {}).items():
        codcur, nome_curso = separar_curso(curso_txt)
        periodos.append({"sigla": sigla, "codcur": codcur,
                         "codhab": codhab_do_curso(codcur, nome_curso),
                         "nome_curso": nome_curso, "periodo_ideal": periodo})

df_requisitos = pd.DataFrame(arestas, columns=[
    "requisito", "disciplina", "tipo_requisito", "codcur", "codhab",
    "nome_curso", "grupo_alternativa", "n_grupos"])
df_periodo = pd.DataFrame(periodos, columns=[
    "sigla", "codcur", "codhab", "nome_curso", "periodo_ideal"])
print(f"{len(df_requisitos)} arestas de requisito | "
      f"{len(df_periodo)} pares (disciplina, curso) com período ideal")
sem_codcur = (df_requisitos["codcur"] == "").sum()
print(f"Arestas sem código de curso: {sem_codcur}")
df_requisitos["tipo_requisito"].value_counts()

1201 arestas de requisito | 914 pares (disciplina, curso) com período ideal
Arestas sem código de curso: 0


tipo_requisito
requisito fraco          758
requisito                396
indicação de conjunto     47
Name: count, dtype: int64

### 6.3 Docentes, turmas, horários e vagas

- **Docentes responsáveis** aparecem na página da disciplina como
  `"NºUSP - Nome"`, um por linha.
- **Docentes ministrantes** aparecem nos horários das turmas, só com o nome. Quando
  uma turma tem vários professores no mesmo horário, as linhas seguintes vêm com dia e
  hora em branco; preenchemos esses buracos com o valor da linha anterior (*forward
  fill*).
- **Vagas:** em algumas turmas, a biblioteca lê a tabela de professores como se fosse
  de vagas (chaves "prof(a)." e nomes de docentes). Mantemos apenas os tipos de vaga
  do Júpiter: obrigatória, optativa eletiva, optativa livre, alunos especiais e
  extracurricular.
- **Turmas sem horário de aula** (estágio, TCC): o Júpiter mostra uma tabela de
  atividades dos professores no lugar dos horários. Essas linhas não entram em
  `horarios`; os professores listados entram em `disciplina_docente` como
  ministrantes da turma.
- **Vagas lidas da página:** quando a turma tem horário e também uma tabela de
  "Atividades Didáticas", a biblioteca lê a tabela errada; nesses casos, as vagas
  são lidas direto do texto da página de oferecimento.
- A marca **"(R)"**, que o Júpiter usa para o professor responsável pela turma, é
  removida do nome.
- Os docentes são unificados pelo nome normalizado (sem acentos e em maiúsculas), o
  que permite ligar "responsável" e "ministrante" à mesma pessoa.

> O oferecimento é um **retrato do semestre corrente** (o que o Júpiter mostra no dia
> da coleta). Por isso a `data_coleta` fica registrada.

In [14]:
RE_DOCENTE = re.compile(r"^\s*(\d{3,9})\s*-\s*(.+?)\s*$")

# Em algumas turmas, a biblioteca lê a tabela de professores como se fosse de
# vagas (chaves "prof(a)." e nomes de docentes). Só estes tipos são vagas de fato.
TIPOS_VAGA = {"obrigatoria", "optativa eletiva", "optativa livre",
              "alunos especiais", "extracurricular"}
CAMPOS_VAGA = ["vagas", "inscritos", "pendentes", "matriculados"]

# Turmas de estágio e TCC não têm horário de aula: no lugar dele, o Júpiter mostra
# uma tabela "Prof(a). | Tipo de Atividade | Carga horária", que a biblioteca lê
# como horário. Linhas cujo dia não é um dia da semana são tratadas assim: o
# cabeçalho é descartado e as demais viram vínculos de docente com a turma.
DIAS_SEMANA = {"seg", "ter", "qua", "qui", "sex", "sab", "dom"}

# Quando a turma tem horário E uma tabela "Atividades Didáticas", a biblioteca lê
# a tabela de atividades no lugar da de vagas. Nesses casos, as vagas são lidas
# direto do texto da página de oferecimento (já em cache).
ROTULO_VAGA = {"Obrigatória": "obrigatoria", "Optativa Eletiva": "optativa eletiva",
               "Optativa Livre": "optativa livre", "Extracurricular": "extracurricular",
               "Alunos Especiais": "alunos especiais"}


def vagas_da_pagina(sigla: str) -> dict:
    soup = obter_soup_cache(f"{URL_BASE}obterTurma?sgldis={sigla}")
    linhas = [l.strip() for l in soup.get_text("\n").splitlines() if l.strip()]
    por_turma, turma, tipo = {}, None, None
    for i, linha in enumerate(linhas):
        if linha == "da Turma:" and i > 0 and linhas[i - 1] == "Código":
            turma, tipo = linhas[i + 1], None
            por_turma[turma] = {}
            continue
        numeros = linhas[i + 1:i + 5]
        if (turma is None or len(numeros) < 4
                or not all(re.fullmatch(r"\d+|-", n) for n in numeros)):
            continue
        valores = dict(zip(CAMPOS_VAGA, (None if n == "-" else int(n)
                                         for n in numeros)))
        if linha in ROTULO_VAGA:
            tipo = ROTULO_VAGA[linha]
            por_turma[turma][tipo] = {**valores, "cursos": {}}
        elif tipo:
            por_turma[turma][tipo]["cursos"][linha] = valores
    return por_turma

vinculos, turmas, horarios, vagas = [], [], [], []
for sigla, dados in bruto.items():
    plano = achatar({k: v for k, v in dados.items() if isinstance(v, (str, dict))
                     and k not in ("requisitos", "periodo ideal")})
    for linha in (campo(plano, "docente") or "").splitlines():
        achado = RE_DOCENTE.match(linha)
        if achado:
            vinculos.append({"sigla": sigla, "nusp_docente": achado.group(1),
                             "nome_docente": achado.group(2),
                             "papel": "responsavel", "turma": None})

    for oferta in dados.get("oferecimento", []):
        turma = oferta["turma"]
        turmas.append({"sigla": sigla, **{k: oferta[k] for k in (
            "turma", "data_inicio", "data_fim", "tipo_turma", "observacoes")}})

        dia = inicio = fim = ""
        for h in oferta["horarios"]:
            if h["dia_semana"] and h["dia_semana"] not in DIAS_SEMANA:
                if h["dia_semana"].lower() != "prof(a).":
                    vinculos.append({"sigla": sigla, "nusp_docente": None,
                                     "nome_docente": limpar_nome(h["dia_semana"]),
                                     "papel": "ministrante", "turma": turma})
                continue
            dia = h["dia_semana"] or dia      # forward fill
            inicio = h["hora_inicio"] or inicio
            fim = h["hora_fim"] or fim
            horarios.append({"sigla": sigla, "turma": turma, "dia_semana": dia,
                             "hora_inicio": inicio, "hora_fim": fim,
                             "professor": limpar_nome(h["professor"]) or None})
            if h["professor"]:
                vinculos.append({"sigla": sigla, "nusp_docente": None,
                                 "nome_docente": limpar_nome(h["professor"]),
                                 "papel": "ministrante", "turma": turma})

        vagas_oferta = oferta["vagas"]
        if not set(vagas_oferta) & TIPOS_VAGA:
            vagas_oferta = vagas_da_pagina(sigla).get(turma, {})
        for tipo_vaga, info in vagas_oferta.items():
            if tipo_vaga not in TIPOS_VAGA:
                continue
            numeros = {k: v for k, v in info.items() if k in CAMPOS_VAGA}
            vagas.append({"sigla": sigla, "turma": turma, "tipo_vaga": tipo_vaga,
                          "curso": "TOTAL", **numeros})
            for curso, nums in info.get("cursos", {}).items():
                nums = {k: v for k, v in nums.items() if k in CAMPOS_VAGA}
                vagas.append({"sigla": sigla, "turma": turma,
                              "tipo_vaga": tipo_vaga, "curso": curso, **nums})

df_vinculos = pd.DataFrame(vinculos, columns=[
    "sigla", "nusp_docente", "nome_docente", "papel", "turma"])
df_vinculos["docente_id"] = df_vinculos["nome_docente"].map(normalizar_nome)

# Tabela de docentes únicos: o NºUSP vem da linha "responsável", quando existir.
# Entre os nomes da mesma pessoa, prefere-se o escrito com maiúsculas
# (as tabelas de atividades de estágio trazem o nome todo em minúsculas).
df_vinculos["_minusculo"] = df_vinculos["nome_docente"].str.islower()
df_docentes = (df_vinculos
               .sort_values(["nusp_docente", "_minusculo"], na_position="last")
               .groupby("docente_id", as_index=False)
               .agg(nome=("nome_docente", "first"),
                    nusp=("nusp_docente", "first"),
                    n_disciplinas=("sigla", "nunique")))
df_vinculos = (df_vinculos
               .drop(columns=["nusp_docente", "nome_docente", "_minusculo"])
               .drop_duplicates())

df_turmas = pd.DataFrame(turmas, columns=[
    "sigla", "turma", "data_inicio", "data_fim", "tipo_turma", "observacoes"])
df_horarios = pd.DataFrame(horarios, columns=[
    "sigla", "turma", "dia_semana", "hora_inicio", "hora_fim", "professor"])
df_vagas = pd.DataFrame(vagas, columns=[
    "sigla", "turma", "tipo_vaga", "curso", *CAMPOS_VAGA])
# "-" no Júpiter significa valor não informado: vira célula vazia, e as colunas
# ficam numéricas (inteiros que aceitam vazio).
df_vagas[CAMPOS_VAGA] = (df_vagas[CAMPOS_VAGA].replace("-", pd.NA)
                         .apply(pd.to_numeric).astype("Int64"))
df_turmas["data_coleta"] = DATA_COLETA

print(f"{len(df_docentes)} docentes | {len(df_vinculos)} vínculos | "
      f"{len(df_turmas)} turmas | {len(df_horarios)} horários | "
      f"{len(df_vagas)} linhas de vagas")

349 docentes | 1221 vínculos | 423 turmas | 587 horários | 1623 linhas de vagas


### 6.4 `bibliografia` — uma linha por referência

O campo de bibliografia é **texto livre** e cada docente formata as referências de
um jeito. Aplicamos uma separação heurística, isto é, regras práticas que funcionam
na maioria dos casos sem garantia de acerto:

- cada linha com mais de 15 caracteres é uma referência (também separamos
  referências escritas na mesma linha com "-" e tabulação);
- descartamos títulos de seção, como "Bibliografia básica:" e "Livros Texto:";
- removemos numerações iniciais (`1.`, `[2]`, `-`);
- `autor_principal` = texto antes da primeira vírgula, ponto e vírgula, parêntese
  ou ponto final (normalmente o sobrenome);
- `ano` = último número de 4 dígitos entre 1900 e 2099.

Isso basta para contar referências e achar livros compartilhados entre
disciplinas. Para a 2ª entrega, vale refinar o casamento (ex.: comparar títulos com
distância de edição).

In [15]:
RE_NUMERACAO = re.compile(r"^\s*(\[\d+\]|\d+[\.\)\-]|[-•*·])\s*")
RE_ANO = re.compile(r"\b(19\d{2}|20\d{2})\b")
# Algumas bibliografias separam referências na mesma linha com "-<tab>".
RE_SEPARADOR = re.compile(r"\n|(?<=\.)\s*-\t")
# Linhas que são títulos de seção ("Bibliografia básica:", "Livros Texto:").
RE_CABECALHO = re.compile(
    r"^(livros?[\s\-]*(texto|base|did[aá]ticos?)?|bibliografia.*|refer[eê]ncias.*|"
    r"leituras?.*|complementar.*|b[aá]sica.*|textos?.*)\s*:?$", re.I)
# Autor principal: o texto até a primeira vírgula, ponto e vírgula, parêntese ou
# ponto final.
RE_FIM_AUTOR = re.compile(r",|;|\s\(|\.\s")

referencias = []
for _, linha_disc in df_disciplinas.iterrows():
    texto = linha_disc["bibliografia_bruta"] or ""
    ordem = 0
    for linha in RE_SEPARADOR.split(texto):
        linha = RE_NUMERACAO.sub("", linha).strip()
        if len(linha) <= 15 or RE_CABECALHO.match(linha):
            continue
        anos = RE_ANO.findall(linha)
        referencias.append({
            "sigla": linha_disc["sigla"],
            "ordem": ordem,
            "referencia": linha,
            "autor_principal": normalizar_nome(RE_FIM_AUTOR.split(linha)[0])[:60],
            "ano": int(anos[-1]) if anos else None,
        })
        ordem += 1

df_biblio = pd.DataFrame(referencias, columns=[
    "sigla", "ordem", "referencia", "autor_principal", "ano"])
print(f"{len(df_biblio)} referências em "
      f"{df_biblio['sigla'].nunique()} disciplinas")

4527 referências em 615 disciplinas


## 7. Etapa 6 — Validação

Antes de salvar, conferimos se o dataset faz sentido. Estas verificações também
podem entrar no relatório, na seção de pré-processamento:

1. **Tamanho**: nenhuma tabela principal pode estar vazia.
2. **Integridade referencial**: toda sigla citada em `requisitos` ou na grade deve
   existir em `disciplinas`. Siglas "órfãs" são, em geral, disciplinas desativadas
   que ainda constam como requisito, ou disciplinas além da profundidade de expansão.
3. **Aciclicidade**: um grafo de pré-requisitos deveria ser um **DAG** (grafo
   direcionado acíclico). Se existisse um ciclo A → B → A, nenhuma das duas
   disciplinas poderia ser cursada. Ciclos indicam erro de coleta ou, mais
   interessante, de cadastro. `indicação de conjunto` é excluída do teste, porque
   ela liga disciplinas cursadas *simultaneamente*.
4. **Inspeção manual**: comparem a SCC0252 com o que aparece no Júpiter.

In [16]:
tabelas = {
    "disciplinas": df_disciplinas, "requisitos": df_requisitos,
    "periodo_ideal": df_periodo, "cursos": df_cursos,
    "grade_curricular": df_grade, "docentes": df_docentes,
    "disciplina_docente": df_vinculos, "oferecimentos": df_turmas,
    "horarios": df_horarios, "vagas": df_vagas, "bibliografia": df_biblio,
}
display(pd.Series({nome: len(df) for nome, df in tabelas.items()},
                  name="linhas").to_frame())

conhecidas = set(df_disciplinas["sigla"])
citadas = set(df_requisitos["requisito"]) | set(df_requisitos["disciplina"])
if not df_grade.empty:
    citadas |= set(df_grade["sigla"])
orfas = sorted(citadas - conhecidas)
print(f"Siglas citadas sem registro em disciplinas: {len(orfas)}", orfas[:15])

dependencias = df_requisitos[
    ~df_requisitos["tipo_requisito"].str.contains("conjunto", na=False)]
grafo = nx.DiGraph()
grafo.add_edges_from(zip(dependencias["requisito"], dependencias["disciplina"]))
print(f"Grafo de requisitos: {grafo.number_of_nodes()} nós, "
      f"{grafo.number_of_edges()} arestas, "
      f"DAG = {nx.is_directed_acyclic_graph(grafo)}")
if not nx.is_directed_acyclic_graph(grafo):
    print("Exemplo de ciclo:", nx.find_cycle(grafo))

,linhas
disciplinas,629
requisitos,1201
periodo_ideal,914
cursos,9
grade_curricular,877
docentes,349
disciplina_docente,1221
oferecimentos,423
horarios,587
vagas,1623


Siglas citadas sem registro em disciplinas: 6 ['7600008', '7600016', 'SLC0601', 'SSC0112', 'SSC0114', 'SSC0143']
Grafo de requisitos: 456 nós, 723 arestas, DAG = True


In [17]:
# Inspeção manual: a SCC0252 nas diferentes tabelas.
SIGLA_TESTE = "SCC0252"
display(df_disciplinas.query("sigla == @SIGLA_TESTE").T.head(12))
display(df_requisitos.query("disciplina == @SIGLA_TESTE"))
display(df_horarios.query("sigla == @SIGLA_TESTE"))

,91
sigla,SCC0252
nome,Visualização Computacional
nome_ingles,Computational Visualization
unidade,Instituto de Ciências Matemáticas e de Computação
departamento,Ciências de Computação
prefixo,SCC
creditos_aula,3
creditos_trabalho,1
carga_horaria_total,75
tipo,Semestral


,requisito,disciplina,tipo_requisito,codcur,codhab,nome_curso,grupo_alternativa,n_grupos
97,SCC0240,SCC0252,requisito fraco,55030,200,Matemática - Habilitação: Bacharelado em Matem...,0,1
98,SCC0250,SCC0252,requisito fraco,55030,200,Matemática - Habilitação: Bacharelado em Matem...,0,1
99,SSC0103,SCC0252,requisito,55041,0,Bacharelado em Ciências de Computação (integral),0,1
100,SCC0504,SCC0252,requisito fraco,55051,4,Bacharelado em Sistemas de Informação (noturno),0,1
101,SCC0204,SCC0252,requisito fraco,55060,0,Bacharelado em Matemática Aplicada e Computaçã...,0,1
102,SSC0800,SCC0252,requisito fraco,55070,None,Bacharelado em Estatística (noturno),0,1
103,SME0827,SCC0252,requisito fraco,55071,4,Bacharelado em Estatística e Ciência de Dados ...,0,1
104,SCC0204,SCC0252,requisito fraco,55090,0,Bacharelado em Ciência de Dados (integral),0,1
105,SCC0604,SCC0252,requisito fraco,97001,0,Engenharia de Computação (integral),0,1


,sigla,turma,dia_semana,hora_inicio,hora_fim,professor
40,SCC0252,2026201,ter,13:30,16:00,Jean Roberto Ponciano


## 8. Salvar as tabelas

Salvamos em CSV (UTF-8). Campos de texto com quebras de linha (ementas,
bibliografia) ficam entre aspas, e o pandas os lê de volta corretamente com
`pd.read_csv`. Para abrir no Excel, usem *Dados → De Texto/CSV* e escolham UTF-8.

**Ao ler as tabelas nos notebooks de visualização**, tratem as colunas de
identificação como texto. Senão, o pandas converte `nusp`, `turma` e `codcur` em
números (e em `float` quando há valores vazios, gerando `1234567.0`):

```python
ids = {"sigla": str, "requisito": str, "disciplina": str, "codcur": str,
       "codhab": str, "nusp": str, "turma": str}
disciplinas = pd.read_csv("dados/processado/disciplinas.csv", dtype=ids)
```

In [18]:
for nome, df in tabelas.items():
    caminho = DIR_FINAL / f"{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8")
    print(f"{caminho}  ({len(df)} linhas, {df.shape[1]} colunas)")

dados\processado\disciplinas.csv  (629 linhas, 29 colunas)
dados\processado\requisitos.csv  (1201 linhas, 8 colunas)
dados\processado\periodo_ideal.csv  (914 linhas, 5 colunas)
dados\processado\cursos.csv  (9 linhas, 5 colunas)
dados\processado\grade_curricular.csv  (877 linhas, 8 colunas)
dados\processado\docentes.csv  (349 linhas, 4 colunas)
dados\processado\disciplina_docente.csv  (1221 linhas, 4 colunas)
dados\processado\oferecimentos.csv  (423 linhas, 7 colunas)
dados\processado\horarios.csv  (587 linhas, 6 colunas)
dados\processado\vagas.csv  (1623 linhas, 8 colunas)
dados\processado\bibliografia.csv  (4527 linhas, 5 colunas)


In [19]:
# # Compacta a pasta e todo o seu conteúdo
# !zip -r dados.zip dados/

# # Faz o download do arquivo compactado
# from google.colab import files
# files.download('dados.zip')

## 9. Dicionário de dados

Use esta seção no relatório (Introdução → informações sobre o conjunto de dados).

**`disciplinas.csv`** — uma linha por disciplina · chave: `sigla`

| Coluna | Tipo | Descrição |
|---|---|---|
| `sigla` | categórico (ID) | código da disciplina (ex.: SCC0252) |
| `nome`, `nome_ingles` | texto | nome da disciplina |
| `unidade`, `departamento`, `prefixo` | categórico | onde a disciplina está alocada |
| `creditos_aula`, `creditos_trabalho`, `creditos_total` | quantitativo discreto | créditos |
| `carga_horaria_total` | quantitativo discreto | horas |
| `tipo` | categórico | semestral, anual etc. |
| `ativacao`, `ano_ativacao`, `desativacao` | temporal | vigência da disciplina |
| `ementa`, `objetivos`, `programa` | texto | ementa, objetivos e conteúdo programático |
| `avaliacao_*` | texto | método, critério e recuperação |
| `bibliografia_bruta` | texto | bibliografia original (básica e complementar juntas) |
| `n_turmas_oferecidas`, `n_cursos_com_requisitos` | quantitativo | contagens |
| `profundidade` | ordinal | 0 = núcleo ICMC; k = trazida no nível k da expansão |
| `no_nucleo` | booleano | se veio da listagem das unidades-alvo |

**`requisitos.csv`** — arestas `requisito → disciplina`

| Coluna | Descrição |
|---|---|
| `requisito`, `disciplina` | siglas de origem e destino |
| `tipo_requisito` | requisito · requisito fraco · indicação de conjunto |
| `codcur`, `codhab`, `nome_curso` | curso (e habilitação) em que o requisito vale; `codhab` vazio para cursos de outras unidades |
| `grupo_alternativa`, `n_grupos` | estrutura "OU de grupos E" |

**`periodo_ideal.csv`** — `sigla`, `codcur`, `codhab`, `nome_curso`, `periodo_ideal`
(ordinal)

**`cursos.csv`** — `codcg` (unidade), `codcur`, `codhab`, `nome_curso`, `url_grade`

**`grade_curricular.csv`** — `codcur`, `codhab`, `sigla`, `nome_na_grade`,
`tipo_vinculo` (obrigatória · optativa eletiva · optativa livre), `semestre_ideal`,
`creditos_aula`, `creditos_trabalho`

**`docentes.csv`** — `docente_id` (nome normalizado), `nome`, `nusp`,
`n_disciplinas`

**`disciplina_docente.csv`** — `sigla`, `docente_id`, `papel` (responsável ·
ministrante), `turma`

**`oferecimentos.csv`** — `sigla`, `turma`, `data_inicio`, `data_fim`, `tipo_turma`,
`observacoes`, `data_coleta`

**`horarios.csv`** — `sigla`, `turma`, `dia_semana`, `hora_inicio`, `hora_fim`,
`professor`

**`vagas.csv`** — `sigla`, `turma`, `tipo_vaga`, `curso` (`TOTAL` = soma da turma)
e as colunas numéricas exibidas pelo Júpiter (vagas, inscritos, pendentes,
matriculados)

**`bibliografia.csv`** — `sigla`, `ordem`, `referencia`, `autor_principal`, `ano`

## 10. Limitações conhecidas

- **Retrato temporal**: oferecimentos, vagas e horários refletem o semestre visível
  no Júpiter na data da coleta, e não um histórico.
- **Currículo vigente**: a grade e os requisitos são os da estrutura curricular
  atual. Mudanças curriculares passadas não aparecem.
- **Bibliografia e nomes**: separação heurística; pode haver referências quebradas
  em duas linhas ou docentes com grafias diferentes não unificados.
- **Expansão limitada**: disciplinas além de `PROFUNDIDADE_MAX_EXPANSAO` aparecem
  como siglas órfãs (ver Etapa 6).
- **Dependência do layout**: se o HTML do Júpiter mudar, o parsing precisa ser
  ajustado. O cache garante que o dataset já coletado continua reprodutível.